# Chapter 14 · Hypothesis Testing: Logic and the Standard Tests: companion notebook
*Data Science and Artificial Intelligence in Health, Volume 1*

> **This notebook accompanies the book chapter; it does not reproduce it.** Definitions, derivations, and explanations are in the book. Here you run the calculations on the case and check the numbers. All data are **synthetic**, and the values are for teaching purposes only.

**Part A** follows the chapter's case, the surgical registry of two hospitals and a quality committee, through the committee's thirteen questions. **Part B** gives the written solution of every chapter exercise (statements are in the book) and, for the exercises whose solution needs a calculation, its code and result. Run the cells in order.

## Part A. The case

**Data.** `data/surgery.csv`: 800 rows, one per operated patient, and eight columns: `patient_id`, `hospital` (A or B), `weekday` (Mon to Sun), `asa_class` (1 to 4), `los_days`, `readmitted` (yes or no), `pain_day1`, `pain_day3`. Each cell carries the number of the book section it belongs to.

### 14.1 Introduction: the data

In [1]:
# 1. Read the registry: one row is one operated patient
import numpy as np
import pandas as pd
from math import sqrt
from scipy import stats

df = pd.read_csv("data/surgery.csv")

# 2. Count rows and columns
rows, columns = df.shape
print("The registry has", rows, "rows (one per patient) and", columns, "columns.")
df.head()                                      # The first five rows

The registry has 800 rows (one per patient) and 8 columns.


,patient_id,hospital,weekday,asa_class,los_days,readmitted,pain_day1,pain_day3
0,P0001,A,Thu,3,4,no,5,6
1,P0002,A,Thu,1,3,no,5,7
2,P0003,B,Mon,2,7,no,6,6
3,P0004,A,Mon,4,4,no,3,1
4,P0005,B,Mon,2,7,yes,7,5


### 14.2 Null and alternative hypotheses

*Does the average stay differ from the target of 5 days, and what must be stated before it can be tested?*

In [2]:
# 1. The sample: length of stay of the 800 patients
los = df["los_days"].values
n = len(los)
m = los.mean()
s = los.std(ddof=1)
print("Patients:", n, "; mean stay", round(m, 3), "days; standard deviation", round(s, 3), "; distance from the target of 5 days:", round(m - 5, 3))

# 2. The hypotheses: H0 is mu = 5; the alternative is two-sided
print("H0: mu = 5 days. Ha: mu different from 5 days (two-sided).")

Patients: 800 ; mean stay 5.344 days; standard deviation 2.69 ; distance from the target of 5 days: 0.344
H0: mu = 5 days. Ha: mu different from 5 days (two-sided).


### 14.3 The test statistic and its null distribution

*How far from 5 days is the sample mean, measured in standard errors?*

In [3]:
# 1. Standard error and the z statistic, taking the sample standard deviation as if it were known
se = s / sqrt(n)
z_stat = (m - 5) / se
print("Standard error:", round(se, 4), "; z =", round(z_stat, 2))

# 2. Skewness and median of the stay
print("Skewness:", round(stats.skew(los), 2), "; median:", np.median(los), "days")

Standard error: 0.0951 ; z = 3.61
Skewness: 1.22 ; median: 5.0 days


### 14.4 The p-value

*How surprising is that distance if the true mean were 5 days?*

In [4]:
# 1. The z statistic and the two-sided and one-sided p-values from the normal curve
p_two = 2 * stats.norm.sf(abs(z_stat))
p_one = stats.norm.sf(z_stat)
print("z =", round(z_stat, 2), "; two-sided p =", round(p_two, 5), "; one-sided p =", round(p_one, 5))

z = 3.61 ; two-sided p = 0.0003 ; one-sided p = 0.00015


### 14.5 Significance level, decisions, and the two types of error

*How often would the committee decide wrongly, and in which direction?*

In [5]:
# 1. 2,000 samples of 100 patients drawn with replacement: p-values testing the true mean and a false mean (5.0)
rng = np.random.default_rng(14)
p_true = []
p_false = []
for repetition in range(2000):
    sample = los[rng.choice(n, 100, replace=True)]
    p_true.append(stats.ttest_1samp(sample, m).pvalue)
    p_false.append(stats.ttest_1samp(sample, 5.0).pvalue)
p_true = np.array(p_true)
p_false = np.array(p_false)

# 2. p-values of the true null in five equal bins from 0 to 1
print("True null, p-values in five equal bins:", np.histogram(p_true, bins=5, range=(0, 1))[0])

# 3. Rejections at alpha = 0.05 when the null is true (type I error) and when it is false
print("Null true: rejected in", round(100 * (p_true < 0.05).mean(), 2), "% of the samples; null false: rejected in", round(100 * (p_false < 0.05).mean(), 2), "%")

# 4. Type II error rate with 100 patients for the 0.34-day difference
print("Type II error rate (beta):", round(100 * (p_false >= 0.05).mean(), 2), "%")

True null, p-values in five equal bins:

 [420 420 370 402 388]
Null true: rejected in 5.55 % of the samples; null false: rejected in 22.25 %
Type II error rate (beta): 77.75 %


### 14.6 Power and the sample size of a test

*How many patients are needed to detect a difference of half a day?*

In [6]:
# 1. Patients for a difference of 0.5 day (alpha = 0.05, power 80%), with the exact normal quantiles
za = stats.norm.ppf(0.975)
zb = stats.norm.ppf(0.80)
n_half = (za + zb) ** 2 * s ** 2 / 0.5 ** 2
print("Patients for 0.5 day:", round(n_half, 1), "-> rounded up", int(np.ceil(n_half)))

# 2. Power with 100 patients for 0.5 day
shift = 0.5 / (s / sqrt(100))
power100 = stats.norm.sf(za - shift) + stats.norm.cdf(-za - shift)
print("Power with 100 patients:", round(100 * power100, 1), "%")

# 3. Patients for the observed difference
n_obs = (za + zb) ** 2 * s ** 2 / (m - 5) ** 2
print("Patients for the observed difference of", round(m - 5, 3), "day:", round(n_obs, 1))

# 4. Patients to detect a rise in readmission from 10% to 15%
n_prop = ((za * sqrt(0.10 * 0.90) + zb * sqrt(0.15 * 0.85)) / 0.05) ** 2
print("Patients for 10% to 15%:", round(n_prop, 1))

# 5. With the t test, which also estimates sigma: the smallest sample whose power reaches 80%
n_t = 10
power_t = 0
while power_t < 0.80:
    n_t = n_t + 1
    t_crit = stats.t.ppf(0.975, n_t - 1)
    shift_t = 0.5 / s * sqrt(n_t)
    power_t = stats.nct.sf(t_crit, n_t - 1, shift_t) + stats.nct.cdf(-t_crit, n_t - 1, shift_t)
print("Patients for the t test:", n_t)

Patients for 0.5 day: 227.2 -> rounded up 228
Power with 100 patients: 46.0 %
Patients for the observed difference of 0.344 day: 480.7
Patients for 10% to 15%: 315.8
Patients for the t test: 230


### 14.7 The one-sample t test for a mean

*Which test serves for the stay when the standard deviation must be estimated?*

In [7]:
# 1. One-sample t test against 5 days
result = stats.ttest_1samp(los, 5)
print("t =", round(result.statistic, 3), "; degrees of freedom", n - 1, "; two-sided p =", round(result.pvalue, 5))

# 2. 95% confidence interval for the mean
multiple = stats.t.ppf(0.975, n - 1)
print("t multiple:", round(multiple, 3), "; 95% CI:", round(m - multiple * se, 3), "to", round(m + multiple * se, 3))
# 3. The z test, taking the standard deviation as known: two-sided p from the normal curve
print("z =", round(z_stat, 3), "; two-sided p from the normal curve =", round(2 * stats.norm.sf(abs(z_stat)), 5))

t = 3.614 ; degrees of freedom 799 ; two-sided p = 0.00032
t multiple: 1.963 ; 95% CI: 5.157 to 5.53
z = 3.614 ; two-sided p from the normal curve = 0.0003


### 14.8 The z test for a proportion

*Does the readmission rate differ from the 10% target?*

In [8]:
# 1. Readmissions and the z test against 10%
k = int((df["readmitted"] == "yes").sum())
p_hat = k / n
z_prop = (p_hat - 0.10) / sqrt(0.10 * 0.90 / n)
print("Readmitted:", k, "of", n, "(", round(100 * p_hat, 2), "%); z =", round(z_prop, 3), "; two-sided p =", round(2 * stats.norm.sf(abs(z_prop)), 4), "; one-sided p =", round(stats.norm.sf(z_prop), 4))

# 2. Exact binomial test and Wilson interval
print("Exact binomial two-sided p =", round(stats.binomtest(k, n, 0.10).pvalue, 4))
centre = (p_hat + za ** 2 / (2 * n)) / (1 + za ** 2 / n)
half = za * sqrt(p_hat * (1 - p_hat) / n + za ** 2 / (4 * n ** 2)) / (1 + za ** 2 / n)
print("Wilson 95%:", round(100 * (centre - half), 1), "to", round(100 * (centre + half), 1), "%")

# 3. Exact binomial power of the z test against 15%, with 316 and 320 patients
for size in (316, 320):
    counts = np.arange(size + 1)
    z_counts = (counts / size - 0.10) / sqrt(0.10 * 0.90 / size)
    rejected = np.abs(z_counts) > za
    print("Exact power with", size, "patients:", round(100 * stats.binom.pmf(counts, size, 0.15)[rejected].sum(), 1), "%")

Readmitted: 95 of 800 ( 11.88 %); z = 1.768 ; two-sided p = 0.0771 ; one-sided p = 0.0385
Exact binomial two-sided p = 0.0871
Wilson 95%: 9.8 to 14.3 %
Exact power with 316 patients: 77.8 %
Exact power with 320 patients: 80.4 %


### 14.9 The two-sample t test: Student and Welch

*Do the two hospitals differ in average stay and in how much it varies?*

In [9]:
# 1. Length of stay in each hospital
a = df[df["hospital"] == "A"]["los_days"].values
b = df[df["hospital"] == "B"]["los_days"].values
print("Hospital A:", len(a), "patients; mean", round(a.mean(), 2), "; standard deviation", round(a.std(ddof=1), 2))
print("Hospital B:", len(b), "patients; mean", round(b.mean(), 2), "; standard deviation", round(b.std(ddof=1), 2))

# 2. Student test (pooled variance)
student = stats.ttest_ind(b, a)
pooled = sqrt(((len(a) - 1) * a.var(ddof=1) + (len(b) - 1) * b.var(ddof=1)) / (len(a) + len(b) - 2))
print("Student: t =", round(student.statistic, 3), "; p =", round(student.pvalue, 5), "; pooled standard deviation", round(pooled, 3), "; standard error", round(pooled * sqrt(1 / len(a) + 1 / len(b)), 4))

# 3. Welch test and 95% interval for the difference
welch = stats.ttest_ind(b, a, equal_var=False)
va = a.var(ddof=1) / len(a)
vb = b.var(ddof=1) / len(b)
df_welch = (va + vb) ** 2 / (va ** 2 / (len(a) - 1) + vb ** 2 / (len(b) - 1))
diff = b.mean() - a.mean()
se_welch = sqrt(va + vb)
margin_welch = stats.t.ppf(0.975, df_welch) * se_welch
print("Welch: t =", round(welch.statistic, 3), "; degrees of freedom", round(df_welch, 1), "; p =", round(welch.pvalue, 5), "; standard error", round(se_welch, 3))
print("Difference B - A:", round(diff, 3), "days; 95% CI", round(diff - margin_welch, 2), "to", round(diff + margin_welch, 2))

Hospital A: 467 patients; mean 5.05 ; standard deviation 2.13
Hospital B: 333 patients; mean 5.75 ; standard deviation 3.28
Student: t = 3.668 ; p = 0.00026 ; pooled standard deviation 2.669 ; standard error 0.1915
Welch: t = 3.427 ; degrees of freedom 528.0 ; p = 0.00066 ; standard error 0.205
Difference B - A: 0.702 days; 95% CI 0.3 to 1.11


In [10]:
# 4. Variance ratio (B over A) and its two-sided p-value
ratio = b.var(ddof=1) / a.var(ddof=1)
df_b = len(b) - 1
df_a = len(a) - 1
p_ratio = 2 * min(stats.f.sf(ratio, df_b, df_a), stats.f.cdf(ratio, df_b, df_a))
print("F =", round(ratio, 3), "; degrees of freedom", df_b, "and", df_a, "; two-sided p =", p_ratio)

# 5. 95% confidence interval for the ratio
print("95% CI for the ratio:", round(ratio / stats.f.ppf(0.975, df_b, df_a), 2), "to", round(ratio * stats.f.ppf(0.975, df_a, df_b), 2))
# 6. Levene test centered on the median, which does not need normal data
levene = stats.levene(b, a, center="median")
print("Levene (median): W =", round(levene.statistic, 1), "; p =", levene.pvalue)

F = 2.367 ; degrees of freedom 332 and 466 ; two-sided p = 1.1334486569800358e-17
95% CI for the ratio: 1.94 to 2.89
Levene (median): W = 56.9 ; p = 1.2250559279692573e-13


### 14.10 The paired t test

*Did pain change between day 1 and day 3?*

In [11]:
# 1. Pilot: the first 30 patients, change in pain from day 1 to day 3
pilot = df.iloc[:30]
change = (pilot["pain_day3"] - pilot["pain_day1"]).values
print("Pilot: mean change", round(change.mean(), 3), "; standard deviation", round(change.std(ddof=1), 3), "; correlation of the two days", round(np.corrcoef(pilot["pain_day1"], pilot["pain_day3"])[0, 1], 2))

# 2. Paired t test and 95% interval
paired = stats.ttest_rel(pilot["pain_day3"], pilot["pain_day1"])
margin = stats.t.ppf(0.975, 29) * change.std(ddof=1) / sqrt(30)
print("Paired: t =", round(paired.statistic, 3), "; p =", round(paired.pvalue, 3), "; 95% CI", round(change.mean() - margin, 2), "to", round(change.mean() + margin, 2))

# 3. The same data treated, wrongly, as two independent groups
unpaired = stats.ttest_ind(pilot["pain_day3"], pilot["pain_day1"])
print("Unpaired: t =", round(unpaired.statistic, 3), "; p =", round(unpaired.pvalue, 3))

# 4. All 800 patients
everyone = stats.ttest_rel(df["pain_day3"], df["pain_day1"])
print("All 800: mean change", round((df["pain_day3"] - df["pain_day1"]).mean(), 3), "; t =", round(everyone.statistic, 2), "; p =", everyone.pvalue)
# 5. 95% interval of the mean change in all 800 patients
change_all = (df["pain_day3"] - df["pain_day1"]).values
margin_all = stats.t.ppf(0.975, 799) * change_all.std(ddof=1) / sqrt(800)
print("All 800: 95% CI", round(change_all.mean() - margin_all, 2), "to", round(change_all.mean() + margin_all, 2))

# 6. Power of the paired test with 30 patients for the change seen in the 800
t_crit_30 = stats.t.ppf(0.975, 29)
shift_30 = abs(change_all.mean()) / change_all.std(ddof=1) * sqrt(30)
power_30 = stats.nct.sf(t_crit_30, 29, shift_30) + stats.nct.cdf(-t_crit_30, 29, shift_30)
print("Power of a paired test with 30 patients for that change:", round(100 * power_30, 1), "%")

Pilot: mean change -0.167 ; standard deviation 1.416 ; correlation of the two days 0.7
Paired: t = -0.645 ; p = 0.524 ; 95% CI -0.7 to 0.36
Unpaired: t = -0.368 ; p = 0.714
All 800: mean change -0.629 ; t = -12.77 ; p = 3.920244399033064e-34
All 800: 95% CI -0.73 to -0.53
Power of a paired test with 30 patients for that change: 66.7 %


### 14.11 The chi-square goodness-of-fit test

*Are admissions spread equally over the weekdays?*

In [12]:
# 1. Admissions from Monday to Friday (weekend admissions are left out)
days = ["Mon", "Tue", "Wed", "Thu", "Fri"]
observed = df[df["weekday"].isin(days)]["weekday"].value_counts().reindex(days).values
expected = observed.sum() / 5
print("Observed:", observed, "; total", observed.sum(), "; expected per day", round(expected, 1))

# 2. Chi-square goodness of fit and the contribution of each day
fit = stats.chisquare(observed)
print("chi-square =", round(fit.statistic, 2), "; degrees of freedom 4; p =", round(fit.pvalue, 4))
print("Contribution of each day:", np.round((observed - expected) ** 2 / expected, 2))

Observed: [198 165 151 135 128] ; total 777 ; expected per day 155.4
chi-square = 19.9 ; degrees of freedom 4; p = 0.0005
Contribution of each day: [11.68  0.59  0.12  2.68  4.83]


### 14.12 The chi-square test of independence and Fisher's exact test

*Is readmission linked to the hospital?*

In [13]:
# 1. Hospital by readmission table
table = pd.crosstab(df["hospital"], df["readmitted"])
print(table)
size_a = table.loc["A"].sum()
size_b = table.loc["B"].sum()
rate_a = table.loc["A", "yes"] / size_a
rate_b = table.loc["B", "yes"] / size_b
print("Readmission rates: A", round(100 * rate_a, 1), "%; B", round(100 * rate_b, 1), "%")

# 2. Chi-square test of independence, without and with the Yates correction
plain = stats.chi2_contingency(table, correction=False)
yates = stats.chi2_contingency(table)
print("chi-square =", round(plain[0], 3), "; p =", round(plain[1], 3), "; with Yates", round(yates[0], 3), "; p =", round(yates[1], 3))
print("Expected counts:", np.round(plain[3], 2).tolist())

# 3. Difference in rates (B - A) and its Newcombe 95% interval
centre_a = (rate_a + za ** 2 / (2 * size_a)) / (1 + za ** 2 / size_a)
half_a = za * sqrt(rate_a * (1 - rate_a) / size_a + za ** 2 / (4 * size_a ** 2)) / (1 + za ** 2 / size_a)
centre_b = (rate_b + za ** 2 / (2 * size_b)) / (1 + za ** 2 / size_b)
half_b = za * sqrt(rate_b * (1 - rate_b) / size_b + za ** 2 / (4 * size_b ** 2)) / (1 + za ** 2 / size_b)
gap_ba = rate_b - rate_a
low_ba = gap_ba - sqrt((rate_b - (centre_b - half_b)) ** 2 + ((centre_a + half_a) - rate_a) ** 2)
high_ba = gap_ba + sqrt(((centre_b + half_b) - rate_b) ** 2 + (rate_a - (centre_a - half_a)) ** 2)
print("Difference B - A:", round(100 * gap_ba, 1), "points; Newcombe 95% CI", round(100 * low_ba, 1), "to", round(100 * high_ba, 1))

# 4. Weekend patients: Fisher's exact test
weekend = df[df["weekday"].isin(["Sat", "Sun"])]
table_weekend = pd.crosstab(weekend["hospital"], weekend["readmitted"])
fisher = stats.fisher_exact(table_weekend)
print("Weekend table (A: no, yes; B: no, yes):", table_weekend.values.tolist(), "; expected", np.round(stats.chi2_contingency(table_weekend, correction=False)[3], 2).tolist())
print("Fisher: odds ratio", round(fisher.statistic, 2), "; p =", round(fisher.pvalue, 3))
# 5. Weekend table: the chi-square with N - 1 in place of N, and Fisher if all three readmissions had been in hospital B
a_no, a_yes = table_weekend.loc["A", "no"], table_weekend.loc["A", "yes"]
b_no, b_yes = table_weekend.loc["B", "no"], table_weekend.loc["B", "yes"]
total_weekend = a_no + a_yes + b_no + b_yes
chi_n1 = (a_yes * b_no - a_no * b_yes) ** 2 * (total_weekend - 1) / ((a_no + a_yes) * (b_no + b_yes) * (a_no + b_no) * (a_yes + b_yes))
print("Chi-square with N - 1:", round(chi_n1, 2), "; p =", round(stats.chi2.sf(chi_n1, 1), 2))
print("Fisher if all three readmissions were in hospital B: p =", round(stats.fisher_exact([[a_no + a_yes, 0], [b_no + b_yes - 3, 3]]).pvalue, 3))

readmitted   no  yes
hospital            
A           418   49
B           287   46
Readmission rates: A 10.5 %; B 13.8 %
chi-square = 2.049 ; p = 0.152 ; with Yates 1.744 ; p = 0.187
Expected counts: [[411.54, 55.46], [293.46, 39.54]]
Difference B - A: 3.3 points; Newcombe 95% CI -1.2 to 8.1
Weekend table (A: no, yes; B: no, yes): [[14, 1], [6, 2]] ; expected [[13.04, 1.96], [6.96, 1.04]]
Fisher: odds ratio 4.67 ; p = 0.269
Chi-square with N - 1: 1.48 ; p = 0.22
Fisher if all three readmissions were in hospital B: p = 0.032


### 14.13 Trend tests across ordered groups

*Is readmission linked to the ASA physical status class?*

In [14]:
# 1. Readmissions by ASA class
df["yes"] = (df["readmitted"] == "yes").astype(int)
events = df.groupby("asa_class")["yes"].sum().values
sizes = df.groupby("asa_class").size().values
print("Readmitted:", events, "of", sizes, "; rates", np.round(100 * events / sizes, 1), "%")

# 2. Trend statistic with scores 1, 2, 3, 4
scores = np.array([1, 2, 3, 4])
total = sizes.sum()
p_bar = events.sum() / total
numerator = (scores * (events - sizes * p_bar)).sum()
spread = (sizes * scores ** 2).sum() - (sizes * scores).sum() ** 2 / total
Z = numerator / sqrt(p_bar * (1 - p_bar) * spread)
print("Z =", round(Z, 2), "; chi-square for trend =", round(Z ** 2, 1), "; p =", 2 * stats.norm.sf(abs(Z)))

# 3. Ordinary chi-square with 3 degrees of freedom
pearson = stats.chi2_contingency(np.array([events, sizes - events]))
print("Chi-square (3 degrees of freedom) =", round(pearson[0], 1), "; p =", pearson[1])

# 4. Difference in readmission between class 4 and class 1, with a Newcombe 95% interval
p4 = events[3] / sizes[3]
p1 = events[0] / sizes[0]
centre4 = (p4 + za ** 2 / (2 * sizes[3])) / (1 + za ** 2 / sizes[3])
half4 = za * sqrt(p4 * (1 - p4) / sizes[3] + za ** 2 / (4 * sizes[3] ** 2)) / (1 + za ** 2 / sizes[3])
centre1 = (p1 + za ** 2 / (2 * sizes[0])) / (1 + za ** 2 / sizes[0])
half1 = za * sqrt(p1 * (1 - p1) / sizes[0] + za ** 2 / (4 * sizes[0] ** 2)) / (1 + za ** 2 / sizes[0])
gap = p4 - p1
low = gap - sqrt((p4 - (centre4 - half4)) ** 2 + ((centre1 + half1) - p1) ** 2)
high = gap + sqrt(((centre4 + half4) - p4) ** 2 + (p1 - (centre1 - half1)) ** 2)
print("Class 4 minus class 1:", round(100 * gap, 1), "points; Newcombe 95% CI", round(100 * low, 1), "to", round(100 * high, 1))

Readmitted: [12 23 40 20] of [185 332 215  68] ; rates [ 6.5  6.9 18.6 29.4] %
Z = 5.91 ; chi-square for trend = 34.9 ; p = 3.3867508421344516e-09
Chi-square (3 degrees of freedom) = 42.2 ; p = 3.6632714093643963e-09
Class 4 minus class 1: 22.9 points; Newcombe 95% CI 12.4 to 34.9


## Part B. Exercise solutions

### 14.15 Exercise 1. Length of stay in hospital A against 5.5 days

The hypotheses are $H_0\colon\mu_A=5.5$ and $H_a\colon\mu_A<5.5$, a one-sided alternative because the question asks only whether the stay is shorter. In the 467 patients of hospital A the mean is 5.051 days and the standard deviation is 2.131, so the standard error is $2.131/\sqrt{467}=0.0986$ and $t=(5.051-5.5)/0.0986=-4.55$ with 466 degrees of freedom. The one-sided p-value is $3.4\times10^{-6}$ (two-sided, $6.9\times10^{-6}$). The 95% confidence interval for the mean is 4.86 to 5.25 days, which does not contain 5.5.

In [15]:
# 1. One-sided t test for hospital A against 5.5 days
a = df[df["hospital"] == "A"]["los_days"].values
res = stats.ttest_1samp(a, 5.5, alternative="less")
se_a = a.std(ddof=1) / sqrt(len(a))
margin_a = stats.t.ppf(0.975, len(a) - 1) * se_a
print("Mean", round(a.mean(), 3), "; standard error", round(se_a, 4), "; t =", round(res.statistic, 2), "; one-sided p =", res.pvalue, "; 95% CI", round(a.mean() - margin_a, 2), "to", round(a.mean() + margin_a, 2))

Mean 5.051 ; standard error 0.0986 ; t = -4.55 ; one-sided p = 3.4470760400203706e-06 ; 95% CI 4.86 to 5.25


### 14.15 Exercise 2. One-sided and two-sided p-values

For $H_0\colon\mu=5$ in all 800 patients, $t=3.61$ with 799 degrees of freedom. The two-sided p-value is 0.00032 and the one-sided p-value for $H_a\colon\mu>5$ is 0.00016, half as large, because only the upper tail counts. The alternative, and so the choice between the two, has to be fixed before the data are seen; choosing the one-sided form after seeing a mean above 5 would halve the p-value to suit the result.

In [16]:
# 1. Two-sided and one-sided p-values against 5 days
print("t =", round(stats.ttest_1samp(los, 5).statistic, 2), "; two-sided p =", round(stats.ttest_1samp(los, 5).pvalue, 5), "; one-sided p =", round(stats.ttest_1samp(los, 5, alternative="greater").pvalue, 5))

t = 3.61 ; two-sided p = 0.00032 ; one-sided p = 0.00016


### 14.15 Exercise 3. P-values under a true and a false null

In 2,000 samples of 100 patients drawn with replacement, testing the true mean (5.344, the mean of the 800 patients; seed 14) gives a p-value below 0.05 in 5.55% of the samples, close to the nominal $\alpha=0.05$: this estimates the type I error rate. Testing the false value 5.0 gives a p-value below 0.05 in 22.25%: this estimates the power of the test with 100 patients for a difference of 0.34 day. Under the true null the p-values are spread uniformly between 0 and 1; under the false null they pile up near 0.

In [17]:
# 1. Share of p-values below 0.05 in the 2,000 samples of 100 patients
print("True mean (", round(m, 3), "):", round(100 * (p_true < 0.05).mean(), 2), "%; false mean (5.0):", round(100 * (p_false < 0.05).mean(), 2), "%")

True mean ( 5.344 ): 5.55 %; false mean (5.0): 22.25 %


### 14.15 Exercise 4. Sample size and power for 0.3 day

With $\sigma=2.690$, $\alpha=0.05$ and 80% power, $n=(1.9600+0.8416)^2\times2.690^2/0.3^2=632$ patients (631.1 before rounding up). With the 800 patients of the registry the power for a difference of 0.3 day is $\Phi(0.3\sqrt{800}/2.690-1.9600)=0.88$. Halving the difference from 0.6 to 0.3 day multiplies the sample size by four, because $n$ is proportional to $1/\delta^{2}$.

In [18]:
# 1. Sample size for 0.3 day and power with the 800 patients
n_03 = (za + zb) ** 2 * s ** 2 / 0.3 ** 2
shift_03 = 0.3 / (s / sqrt(800))
print("Patients:", round(n_03, 1), "; power with 800 patients:", round(stats.norm.sf(za - shift_03) + stats.norm.cdf(-za - shift_03), 3))

Patients: 631.1 ; power with 800 patients: 0.884


### 14.15 Exercise 5. Sample size for a proportion

For 10% against 13%, $n=\bigl(1.9600\sqrt{0.10\times0.90}+0.8416\sqrt{0.13\times0.87}\bigr)^{2}/0.03^{2}=843$ patients (842.99 before rounding up). For 10% against 15% it is 316, so a difference of 3 points needs 2.7 times as many patients as one of 5 points. The sample size is roughly proportional to $1/\delta^{2}$ and $(5/3)^{2}=2.8$; the small remainder comes from the variance of the proportion, which is larger at 13% than at 15%.

In [19]:
# 1. Sample size for 10% against 13%, compared with 10% against 15%
n_13 = ((za * sqrt(0.10 * 0.90) + zb * sqrt(0.13 * 0.87)) / 0.03) ** 2
print("Patients: 10% to 13%:", round(n_13, 1), "; 10% to 15%:", round(n_prop, 1), "; ratio", round(n_13 / n_prop, 2))

Patients: 10% to 13%: 843.0 ; 10% to 15%: 315.8 ; ratio 2.67


### 14.15 Exercise 6. Readmission in hospital B against 10\%

Hospital B readmitted 46 of 333 patients, 13.8%. The z test gives $z=(0.1381-0.10)/\sqrt{0.10\times0.90/333}=2.32$ and a two-sided $p=0.020$ (one-sided, $0.010$). The exact binomial test gives a two-sided $p=0.028$ (one-sided, $0.016$). Both reject at 0.05 for the two-sided alternative. The exact test does not rely on the normal approximation and is the more cautious of the two; with $np_0=33$ expected readmissions both are acceptable, and the difference between them (0.020 against 0.028) shows how much the approximation can move the p-value near the threshold.

In [20]:
# 1. Hospital B against 10%: z test and exact binomial test
k_b = int((df[df["hospital"] == "B"]["readmitted"] == "yes").sum())
n_b = int((df["hospital"] == "B").sum())
z_hb = (k_b / n_b - 0.10) / sqrt(0.10 * 0.90 / n_b)
print("Hospital B:", k_b, "of", n_b, "; z =", round(z_hb, 2), "; two-sided p =", round(2 * stats.norm.sf(abs(z_hb)), 4), "; one-sided p =", round(stats.norm.sf(z_hb), 4))
print("Exact binomial: two-sided p =", round(stats.binomtest(k_b, n_b, 0.10).pvalue, 4), "; one-sided p =", round(stats.binomtest(k_b, n_b, 0.10, alternative="greater").pvalue, 4))

Hospital B: 46 of 333 ; z = 2.32 ; two-sided p = 0.0203 ; one-sided p = 0.0102
Exact binomial: two-sided p = 0.0277 ; one-sided p = 0.0159


### 14.15 Exercise 7. Student and Welch for the two hospitals

Student: $t=3.668$ with 798 degrees of freedom, $p=0.00026$, standard error 0.1915, 95% confidence interval 0.33 to 1.08 days. Welch: $t=3.427$ with 528.0 degrees of freedom, $p=0.00066$, standard error 0.205, 95% confidence interval 0.30 to 1.11 days. Both reject. They differ because the Student test pools the two variances ($s_p=2.669$), while the group standard deviations are 2.13 and 3.28 and the larger group (A) has the smaller variance; pooling gives the larger variance too little weight and the standard error comes out too small. The Welch interval is wider and is the one to report.

In [21]:
# 1. Student and Welch tests with the interval for the difference
se_student = pooled * sqrt(1 / len(a) + 1 / len(b))
margin_student = stats.t.ppf(0.975, len(a) + len(b) - 2) * se_student
print("Student: p =", round(student.pvalue, 5), "; 95% CI", round(diff - margin_student, 2), "to", round(diff + margin_student, 2))
print("Welch: p =", round(welch.pvalue, 5), "; 95% CI", round(diff - margin_welch, 2), "to", round(diff + margin_welch, 2))

Student: p = 0.00026 ; 95% CI 0.33 to 1.08
Welch: p = 0.00066 ; 95% CI 0.3 to 1.11


### 14.15 Exercise 8. Equality of two variances

The variance ratio is $F=s_B^{2}/s_A^{2}=10.75/4.54=2.37$ (hospital B over A). How to calculate it: (1) take the variance of each group, $s_B^{2}=10.75$ (333 patients) and $s_A^{2}=4.54$ (467 patients); (2) divide the variance of B by that of A; under the null hypothesis of equal variances and with normal data, the ratio follows the F distribution with $333-1=332$ and $467-1=466$ degrees of freedom (numerator group first); (3) the two-sided p-value is twice the smaller tail of that distribution, here far below 0.001; (4) the 95% confidence interval for the ratio of the two population variances divides $F$ by the 97.5th percentile of the F distribution (1.219) for the lower limit and by its 2.5th percentile (0.818) for the upper limit: $2.37/1.219=1.94$ and $2.37/0.818=2.89$. The interval does not contain 1, so the variance of hospital B is estimated to be between 1.9 and 2.9 times that of A. The F distribution is available as `scipy.stats.f` (`f.sf`, `f.cdf` and `f.ppf`). The F test is sensitive to non-normality, and length of stay is skewed, so its p-value and its interval are approximate; the Levene test centered on the median, which does not need normality, also gives $p<0.001$ ($W=56.9$) but does not validate the interval. It should not decide between the Student and Welch tests, because the choice made after a preliminary test distorts the error rates of the test that follows, and the Welch test is safe when the variances are equal.

In [22]:
# 1. Variance ratio and its 95% interval
print("F =", round(ratio, 3), "; two-sided p =", p_ratio, "; 95% CI", round(ratio / stats.f.ppf(0.975, df_b, df_a), 2), "to", round(ratio * stats.f.ppf(0.975, df_a, df_b), 2))

F = 2.367 ; two-sided p = 1.1334486569800358e-17 ; 95% CI 1.94 to 2.89


### 14.15 Exercise 9. Pain: paired and unpaired

In the first 30 patients the mean change from day 1 to day 3 is $-0.17$ point with a standard deviation of 1.42. Paired: $t=-0.645$ with 29 degrees of freedom, $p=0.524$, 95% confidence interval $-0.70$ to $0.36$. Unpaired: $t=-0.37$, $p=0.71$. The paired test is the right one: the same patient is measured twice, and the correlation between the two scores is 0.70, so the differences vary much less than the scores. The unpaired test ignores this and treats the two days as independent groups. In all 800 patients the mean change is $-0.63$ point (95% confidence interval $-0.73$ to $-0.53$), with $t=-12.77$ and $p<0.001$. The pilot of 30 patients was inconclusive, not evidence that there was no change: its interval includes $-0.63$, and a paired test with 30 patients would have a power of about 67% for a fall of that size.

In [23]:
# 1. Pilot of 30 and all 800 patients: paired and unpaired tests
print("Pilot (30): paired p =", round(paired.pvalue, 3), "; unpaired p =", round(unpaired.pvalue, 3), "; correlation", round(np.corrcoef(pilot["pain_day1"], pilot["pain_day3"])[0, 1], 2))
print("All 800: paired p =", everyone.pvalue, "; t =", round(everyone.statistic, 2))

Pilot (30): paired p = 0.524 ; unpaired p = 0.714 ; correlation 0.7
All 800: paired p = 3.920244399033064e-34 ; t = -12.77


### 14.15 Exercise 10. Admissions by weekday

The 777 weekday admissions are 198, 165, 151, 135 and 128 from Monday to Friday, against 155.4 expected for each. The statistic is $\chi^{2}=19.9$ with 4 degrees of freedom and $p=0.00052$. Monday contributes the most, $(198-155.4)^2/155.4=11.68$, followed by Friday (4.83) and Thursday (2.68). The data are evidence that the five days are not equal; the test does not say why.

In [24]:
# 1. Chi-square of each weekday
print("Weekday contributions:", np.round((observed - expected) ** 2 / expected, 2), "; chi-square =", round(fit.statistic, 2), "; largest:", days[int(np.argmax((observed - expected) ** 2 / expected))])

Weekday contributions: [11.68  0.59  0.12  2.68  4.83] ; chi-square = 19.9 ; largest: Mon


### 14.15 Exercise 11. Hospital and readmission; Fisher at weekends

The expected count of each cell is row total times column total divided by 800: 411.5 and 55.5 in hospital A, 293.5 and 39.5 in hospital B. Without the continuity correction, $\chi^{2}=\sum(O-E)^{2}/E=2.049$ with 1 degree of freedom and $p=0.152$. The continuity correction (Yates) exists because the counts are whole numbers while the chi-square curve is continuous: it moves each observed count half a unit toward its expected count, $\chi^{2}=\sum(|O-E|-0.5)^{2}/E$. Here every cell has $|O-E|=6.46$, so the four terms $(6.46-0.5)^{2}/E$ are 0.086, 0.640, 0.121 and 0.897, which add to $\chi^{2}=1.744$ and $p=0.187$ (in Python, `chi2_contingency` with `correction=True`). The correction makes the test more conservative, and with 800 patients it does not change the conclusion. The readmission rates are 10.5% (49/467) and 13.8% (46/333), a difference of 3.3 percentage points. To build the Newcombe 95% interval of this difference, first calculate the Wilson interval of each rate (A: 8.0% to 13.6%; B: 10.5% to 17.9%); then, with $d=p_B-p_A$, $p$ the rate of each hospital and $l$ and $u$ its Wilson limits, the lower limit is $d-\sqrt{(p_B-l_B)^{2}+(u_A-p_A)^{2}}$ and the upper limit is $d+\sqrt{(u_B-p_B)^{2}+(p_A-l_A)^{2}}$, which gives $-1.2$ to $8.1$. The data do not show a difference. Among the 23 weekend patients the counts are 1/15 in hospital A and 2/8 in B; the expected counts of readmission are 1.96 and 1.04, below 5, so the chi-square approximation is unreliable. Fisher's exact test lists every way to place the 3 readmissions among the 23 patients with the hospital totals (15 and 8) fixed: the probability of $a$ readmissions in hospital A is $\binom{15}{a}\binom{8}{3-a}/\binom{23}{3}$, which gives 0.032 for $a=0$, 0.237 for $a=1$ (the observed table), 0.474 for $a=2$ and 0.257 for $a=3$. The p-value adds the probabilities of all tables no more probable than the observed one: $0.032+0.237=0.269$. The odds ratio is $(2/6)/(1/14)=4.67$, which with 23 patients is compatible with any association from none to a large one.

In [25]:
# 1. Independence with and without correction, and Fisher at weekends
print("chi-square, plain:", round(plain[0], 3), "; p =", round(plain[1], 3), "; Yates:", round(yates[0], 3), "; p =", round(yates[1], 3), "; Fisher p =", round(fisher.pvalue, 3))

chi-square, plain: 2.049 ; p = 0.152 ; Yates: 1.744 ; p = 0.187 ; Fisher p = 0.269


### 14.15 Exercise 12. Trend in readmission across ASA classes

The readmission rates are 12/185 (6.5%), 23/332 (6.9%), 40/215 (18.6%) and 20/68 (29.4%) for classes 1 to 4. With scores 1, 2, 3, 4 the trend statistic is $Z=5.91$, $\chi^{2}_{\text{trend}}=34.9$ with 1 degree of freedom and $p=3.4\times10^{-9}$. The ordinary chi-square test for 3 degrees of freedom gives 42.2. The trend statistic is smaller because it tests one specific pattern, a linear change in the log odds with the class, but its p-value is as small and it answers the more specific question. The 3-degree test is useful when no order is assumed; the trend test when the groups are ordered and the question is about a gradient. The trend test does not show that the rates rise by equal steps. Neither shows that the class causes the readmission.

In [26]:
# 1. Trend and 3-degree chi-square
print("Trend chi-square:", round(Z ** 2, 1), "; 3 degrees of freedom:", round(pearson[0], 1), "; p of trend =", 2 * stats.norm.sf(abs(Z)))

Trend chi-square: 34.9 ; 3 degrees of freedom: 42.2 ; p of trend = 3.3867508421344516e-09


---
*Synthetic data (seed 14, `generate_data.py`). Teaching values only; nothing here is an estimate for a real hospital.*